In [33]:
%load_ext autoreload
%autoreload 2
%reset -f

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
import sys
import os
from pathlib import Path

# Add directory containing locallib to Python path
# Find the directory containing locallib
os.chdir('/home/sandbox/personal-repos/DA-3515/standalone')
current_dir = Path(os.getcwd())
docker_dir = Path('/home/sandbox/personal-repos/DA-3515/standalone')

# Check if locallib exists in current directory
if (current_dir / 'locallib').exists():
    notebook_dir = current_dir
elif docker_dir.exists() and (docker_dir / 'locallib').exists():
    notebook_dir = docker_dir
else:
    # Fallback: use current working directory
    notebook_dir = current_dir

# Add the directory containing locallib to sys.path
notebook_dir_str = str(notebook_dir.resolve())
if notebook_dir_str not in sys.path:
    sys.path.insert(0, notebook_dir_str)


from locallib.query import get_users, Query, get_surveys
from locallib.picarrodb import *
from locallib.box import *
from locallib.slack import *
import logging

from datetime import datetime, date, timedelta
import pandas as pd
import os
from openpyxl import load_workbook
from openpyxl.utils.dataframe import dataframe_to_rows
from openpyxl.utils.datetime import from_excel, to_excel
from openpyxl.pivot import fields as xl_pivot_fields


In [35]:
os.chdir('/home/sandbox/personal-repos/DA-3564')

In [36]:
BOX_FOLDER_ID = 372723239821
OUTPUT_FILE = 'CADENT_SurveyTracker.xlsx'

OUT_COLS = ['UserName', 'SurveyorUnit', 'AnalyzerSerialNumber', 'SurveyTag', 'StartDateTimeSurvey', 'EndDateTimeSurvey', 'Driver', 'StabilityClass', 'Status', 'DrivingLengthKM', 'RawDurationMinutes', 'ReportId', 'AnalysisDate']

In [ ]:
logger = logging.getLogger('DA-3564')
logger.setLevel(logging.INFO)
SLACK_CHANNEL = 'C0AMJMA1U13'
IO_SLACK = SlackWriter(SLACK_CHANNEL)

# Clear existing handlers to prevent duplicates
logger.handlers.clear()

# Create a custom handler using IOSlack
slack_handler = logging.StreamHandler(IO_SLACK)
slack_handler.setLevel(logging.INFO)
slack_formatter = logging.Formatter('%(name)s - %(asctime)s - %(message)s', datefmt='%d-%m-%y - %H:%M:%S')
slack_handler.setFormatter(slack_formatter)

# Create a file handler
file_handler = logging.FileHandler('DA-3564.log')
file_handler.setLevel(logging.INFO)
file_formatter = logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s')
file_handler.setFormatter(file_formatter)

# Add both handlers to the logger
logger.addHandler(slack_handler)
logger.addHandler(file_handler)

In [ ]:
logger.info('=' * 20)
logger.info('Starting Process DA-3564')
logger.info('Cadent Survey Tracker')
logger.info('Getting Users')
a = Query(get_users(customer_name = 'Cadent', user_table = '#TempUsers'))
logger.info('Getting Surveys')
a.set_child(Query(get_surveys(user_table = '#TempUsers', start_date = '2026-01-01')))
surveys = a.execute(EU2_Conn)

INFO:DA-3564:Starting Process DA-3564
INFO:DA-3564:Cadent Survey Tracker
INFO:DA-3564:Getting Users
INFO:DA-3564:Getting Surveys


In [39]:
logger.info('Getting the surveys with reports')
surveys.db.set_query('SELECT ReportId, SurveyId FROM ReportDrivingSurvey JOIN Report ON ReportDrivingSurvey.ReportId = Report.Id WHERE SurveyId IN (SELECT SurveyId FROM #TempSurveys)')
df = surveys.db.execute(EU2_Conn, source_col = 'SurveyId', temp_table_name = '#TempSurveys', erase_table = True)

INFO:DA-3564:Getting the surveys with reports
/home/sandbox/personal-repos/locallib_packages/locallib/picarrodb/PConnection.py:75: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  except Exception as e:


In [40]:
logger.info('Merging surveys with reports')
surveys_wreport = pd.merge(surveys, df, on='SurveyId', how='left').sort_values(by='StartDateTimeSurvey')

logger.info('Adding Analysis Date')
surveys_wreport['Date'] = surveys_wreport.StartDateTimeSurvey.dt.date
surveys_wreport['Hour'] = surveys_wreport['StartDateTimeSurvey'].dt.hour
for index, row in surveys_wreport.iterrows():
    if (row['Hour'] >= 0) & (row['Hour'] <= 8):
        surveys_wreport.at[index, 'AnalysisDate'] = row['Date'] - timedelta(days=1)
    else:
        surveys_wreport.at[index, 'AnalysisDate'] = row['Date']

INFO:DA-3564:Merging surveys with reports
INFO:DA-3564:Adding Analysis Date


In [ ]:
logger.info('Creating Output DataFrame')
out_df = surveys_wreport[OUT_COLS]
out_df['DrivingLengthKM'] = out_df['DrivingLengthKM'].apply(lambda x: round(float(x), 1) if pd.notnull(x) else x)
# Load existing workbook (do not overwrite existing sheets)
wb = load_workbook(OUTPUT_FILE)

# Remove sheet named 'RawData' if it exists, to overwrite its content
if 'RawData' in wb.sheetnames:
    std = wb['RawData']
    wb.remove(std)

# Create new 'RawData' sheet
ws = wb.create_sheet('RawData')

# Write DataFrame to the 'RawData' sheet
for r_idx, row in enumerate(dataframe_to_rows(out_df, index=False, header=True), 1):
    for c_idx, value in enumerate(row, 1):
        ws.cell(row=r_idx, column=c_idx, value=value)

ws_pivot = wb['Dashboard']
pivot = ws_pivot._pivots[0]

logger.info('Saving the workbook')
wb.save(OUTPUT_FILE)

logger.info('Uploading to Box')
box_folder = BoxFile(OUTPUT_FILE, BOX_FOLDER_ID)
box_folder.upload()

logger.info('Process Completed')
logger.info('=' * 20)

INFO:DA-3564:Creating Output DataFrame
/tmp/ipykernel_36825/960206631.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  out_df['DrivingLengthKM'] = out_df['DrivingLengthKM'].apply(lambda x: round(float(x), 1) if pd.notnull(x) else x)
INFO:DA-3564:Saving the workbook
INFO:DA-3564:Uploading to Box
INFO:DA-3564:Process Completed
